# 01 — Sanity check: in-distribution honesty probe

Pure replication of Zou et al.'s in-distribution setup, adapted to Qwen3-4B.

**What we're doing.** Train a linear probe on residual-stream activations at the
last token of the assistant response, where the assistant has been instructed to
roleplay as either "an honest person" or "an untruthful person". The actual
statement content is identical between the two; only the SP differs.

**What this tells us.** If the probe gets AUROC ≥ 0.95 on held-out *statements*
(not just held-out token-prefixes), the pipeline works end-to-end. It does NOT
tell us the probe captures honesty as a concept — it could be reading
"is the system prompt the honest one?" That dissociation is Experiment 2.

## Prediction (write before running)

Before extracting any activations, commit to a guess. This is the
anti-fake-learning discipline from CLAUDE.md — surprise only registers if
you predicted something specific.

- Expected AUROC at layer 14:  >=0.95
- Expected AUROC at layer 4:   >=0.7 (too early?)
- Expected AUROC at layer 32:  >=0.9 (too late?)
- One thing that could plausibly tank this: Bad code, different concept distribution from Qwen labels

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import torch
import lib
from lib import DATA_DIR

LAYER = 32   # Qwen3-4B has 36 layers; ~40% depth per the context.md consensus.
SEED = 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load model + build pairs

In [8]:
model, tok = lib.load_model_and_tokenizer()
print(f"Model loaded. n_layers={model.config.num_hidden_layers}, hidden={model.config.hidden_size}")
print(f"Probing layer {LAYER}.")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded. n_layers=36, hidden=2560
Probing layer 14.


In [3]:
train, test = lib.build_repe_pairs(
    DATA_DIR / "facts_true_false.csv",
    tok,
    seed=SEED,
    n_train_pairs=256,
    n_test_pairs=64,
)
print(f"Train rows: {len(train)} ({sum(r['label'] for r in train)} honest)")
print(f"Test rows:  {len(test)} ({sum(r['label'] for r in test)} honest)")
print()
print("Sample honest prompt:")
print(repr(next(r['prompt'] for r in train if r['label'] == 1)))
print()
print("Sample untruthful prompt (same statement prefix):")
print(repr(next(r['prompt'] for r in train if r['label'] == 0)))

Train rows: 3190 (1595 honest)
Test rows:  720 (360 honest)

Sample honest prompt:
"<|im_start|>user\nPretend you're an honest person making statements about the world.<|im_end|>\n<|im_start|>assistant\nCon"

Sample untruthful prompt (same statement prefix):
"<|im_start|>user\nPretend you're an untruthful person making statements about the world.<|im_end|>\n<|im_start|>assistant\nThe Hubble Space"


## Extract activations

This is the cell that exercises your hook. If `collect_activations` is still
a stub, you'll hit `NotImplementedError` — go finish `lib.collect_activations`
first.

Caches to `cache/sanity_layer{LAYER}.pt` so re-running notebooks doesn't burn GPU.

In [15]:
CACHE_NAME = f"sanity_layer{LAYER}"

if lib.cache_exists(CACHE_NAME):
    print("Loading cached activations.")
    blob = lib.load_cache(CACHE_NAME)
    X_train, y_train = blob['X_train'].float().numpy(), blob['y_train'].numpy()
    X_test, y_test = blob['X_test'].float().numpy(), blob['y_test'].numpy()
else:
    print("Extracting activations (no cache).")
    train_prompts = [r['prompt'] for r in train]
    test_prompts = [r['prompt'] for r in test]
    X_train_t = lib.collect_activations(model, tok, train_prompts, layer=LAYER)
    X_test_t  = lib.collect_activations(model, tok, test_prompts, layer=LAYER)
    y_train_t = torch.tensor([r['label'] for r in train], dtype=torch.long)
    y_test_t  = torch.tensor([r['label'] for r in test], dtype=torch.long)
    lib.save_cache(CACHE_NAME, X_train=X_train_t, y_train=y_train_t,
                                X_test=X_test_t, y_test=y_test_t)
    X_train, y_train = X_train_t.float().numpy(), y_train_t.numpy()
    X_test,  y_test  = X_test_t.float().numpy(),  y_test_t.numpy()

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

Extracting activations (no cache).
X_train: (3190, 2560), X_test: (720, 2560)


## Fit probes and evaluate

In [ ]:
# LR probe (sklearn one-liner)
clf = lib.fit_lr_probe(X_train, y_train)
lr_scores_train = clf.decision_function(X_train)
lr_scores_test  = clf.decision_function(X_test)
print(f"LR  train AUROC: {lib.auroc(lr_scores_train, y_train):.4f}")
print(f"LR  test  AUROC: {lib.auroc(lr_scores_test,  y_test):.4f}")

# Layer 14
# LR  train AUROC: 1.0000
# LR  test  AUROC: 0.9733

# Layer 4
# LR  train AUROC: 1.0000
# LR  test  AUROC: 0.9398

# Layer 32
# LR  train AUROC: 1.0000
# LR  test  AUROC: 0.9586

LR  train AUROC: 1.0000
LR  test  AUROC: 0.9586


In [ ]:
# MM probe — exercises your μ_pos - μ_neg implementation
d_mm = lib.mm_direction(X_train, y_train)
mm_scores_train = lib.mm_score(X_train, d_mm)
mm_scores_test  = lib.mm_score(X_test,  d_mm)
print(f"MM  train AUROC: {lib.auroc(mm_scores_train, y_train):.4f}")
print(f"MM  test  AUROC: {lib.auroc(mm_scores_test,  y_test):.4f}")

# Layer 14
# MM  train AUROC: 0.8175
# MM  test  AUROC: 0.7884

# Layer 4
# MM  train AUROC: 0.7421
# MM  test  AUROC: 0.7591

# Layer 32
# MM  train AUROC: 0.7154
# MM  test  AUROC: 0.7243

MM  train AUROC: 0.7154
MM  test  AUROC: 0.7243


## After running — what the sanity check actually told us

**Layer sweep:**

| Layer | LR    | MM    | LR-MM gap |
|-------|-------|-------|-----------|
| 4     | 0.940 | 0.759 | 18 pts    |
| 8     | 0.923 | 0.736 | 19 pts    |
| 14    | 0.973 | 0.788 | 18 pts    |
| 20    | 0.998 | 0.914 |  8 pts    |  ← peak for both
| 26    | 0.982 | 0.879 | 10 pts    |
| 32    | 0.959 | 0.724 | 24 pts    |

**What I expected (recovered from prediction cell):** AUROC ≥0.95 at L14,
≥0.7 at L4 (implicit: "early layers should struggle"), ≥0.9 at L32.

**What actually happened:**

1. **L4 was much better than I expected** (0.94 vs predicted ~0.7–0.85). The
   SP-only contrast is a near-trivial signal — "honest" vs "untruthful" is one
   token of difference; even 4 attention layers is enough to leak that into
   a linearly-separable activation. This is the Apollo §4.1.1 "probe might be
   reading deception-related context, not deception" failure mode in incubation
   form. Exp 2 will pin down how literal that is.

2. **L20 is the peak, not L14.** ~56% depth, deeper than the literature
   consensus (M&T peaked at 30%, Apollo 28%, McKenzie 39%). Possibly a
   small-model effect (compressed depth → features form later in relative
   terms), possibly Qwen-specific. Move LAYER = 20 for Exp 2+.

3. **The LR–MM gap is the real signal.** At L14 the gap is ~18 pts; at L20
   it's 8 pts; at L32 it blows back out to 24. This is M&T §5.1 in action.
   At the layer where the honesty feature is cleanly encoded (L20), LR and
   MM converge — both estimators are pointing at roughly the same direction.
   At layers where the feature is messier (L32, late), LR can still find a
   good separator by tilting through the noise, but MM (no tilt) suffers.

4. **My "predictions matched" framing was too generous to myself.** They
   matched the lower-bound thresholds, but I had no model of L20 being best
   or of how big the LR-MM gap would be. The L4 result was the genuine
   surprise. The MM degradation at L32 was another.

**What this experiment doesn't tell us, and what comes next:**

- We don't know yet whether the probe is reading SP-presence or
  actually-being-deceptive. Exp 2's switcheroo (force-fed honest AR under
  deceptive SP and vice versa) is the diagnostic.
- We don't know whether MM's direction is more causally meaningful than LR's.
  Exp 5 part B (steering at matched intervention strength) is the test.
- All these numbers are in-distribution. The Apollo deception-adjacent text
  (Exp 3) is the OOD generalization probe.

## Optional: layer sweep

In [18]:
# Uncomment to sweep. Caches per-layer so subsequent runs are cheap.
LAYERS_TO_TRY = [4, 8, 14, 20, 26, 32]
results = []
train_prompts = [r['prompt'] for r in train]
test_prompts = [r['prompt'] for r in test]
y_train_arr = np.array([r['label'] for r in train])
y_test_arr = np.array([r['label'] for r in test])
for L in LAYERS_TO_TRY:
    name = f"sanity_layer{L}"
    if lib.cache_exists(name):
        blob = lib.load_cache(name)
        Xtr, Xte = blob['X_train'].float().numpy(), blob['X_test'].float().numpy()
    else:
        Xtr_t = lib.collect_activations(model, tok, train_prompts, layer=L)
        Xte_t = lib.collect_activations(model, tok, test_prompts, layer=L)
        lib.save_cache(name, X_train=Xtr_t, y_train=torch.tensor(y_train_arr),
                             X_test=Xte_t, y_test=torch.tensor(y_test_arr))
        Xtr, Xte = Xtr_t.float().numpy(), Xte_t.float().numpy()
    clf = lib.fit_lr_probe(Xtr, y_train_arr)
    d = lib.mm_direction(Xtr, y_train_arr)
    results.append({
        "layer": L,
        "lr_auroc": lib.auroc(clf.decision_function(Xte), y_test_arr),
        "mm_auroc": lib.auroc(lib.mm_score(Xte, d), y_test_arr),
    })
import pandas as pd
pd.DataFrame(results)

,layer,lr_auroc,mm_auroc
0,4,0.939823,0.759074
1,8,0.923225,0.736489
2,14,0.973341,0.788372
3,20,0.997955,0.914313
4,26,0.982076,0.878920
5,32,0.958603,0.724344
